# Playground

In [17]:
import geopandas as gpd
from shapely import geometry, set_precision
import shapely
from shapely.ops import unary_union
import networkx as nx
import momepy

## Validate CRS function

In [3]:
def validate_crs(gdf):
    if gdf.crs.is_geographic:
        gdf = gdf.to_crs(epsg=4326)
        return gdf.to_crs(gdf.estimate_utm_crs())
    return gdf

## Centroid function

In [4]:
def get_centroids(gdf):
    gdf = validate_crs(gdf)
    gdf['centroid'] = gdf.geometry.centroid
    return gdf

In [14]:
gdf_shape = gpd.read_file('test_shapes.gpkg')
get_centroids(gdf_shape)

,id,feat,geometry,centroid
0,1,first,"MULTIPOLYGON (((322109.431 4730357.405, 322255...",POINT (321816.211 4730103.326)
1,2,second,"MULTIPOLYGON (((322535.551 4730198.913, 322504...",POINT (322622.382 4730293.955)
2,3,third,"MULTIPOLYGON (((322285.486 4729706.411, 322224...",POINT (322362.492 4729893.569)


## Segment function

In [10]:
def segment_lines(gdf_polylines):
    gdf_polylines = validate_crs(gdf_polylines)
    segmented_lines = unary_union(gdf_polylines.geometry)
    segmented_gdf = gpd.GeoDataFrame(geometry=gpd.GeoSeries(list(segmented_lines.geoms)), crs=gdf_polylines.crs)
    segmented_gdf['length'] = segmented_gdf.geometry.length
    return segmented_gdf

In [13]:
gdf_lines = gpd.read_file('test_lines2.gpkg')
segment_lines(gdf_lines) #.to_file('test_lines_result.gpkg')

,geometry,length
0,"LINESTRING (322140.932 4730394.071, 322234.37 ...",136.476794
1,"LINESTRING (322234.37 4730493.546, 322314.936 ...",537.268139
2,"LINESTRING (322462.129 4730299.918, 322298.947...",1545.088857
3,"LINESTRING (321449.876 4730126.306, 321489.203...",215.714815
4,"LINESTRING (321489.203 4730338.405, 321507.324...",99.396898
5,"LINESTRING (321446.182 4730344.025, 321489.203...",43.386815
6,"LINESTRING (321489.203 4730338.405, 321624.421...",136.366580
7,"LINESTRING (321624.421 4730320.743, 321781.21 ...",851.994928
8,"LINESTRING (322234.37 4730493.546, 322462.129 ...",298.942138
9,"LINESTRING (322462.129 4730299.918, 322474.402...",16.107563


## Create isochrones function

In [19]:
def create_isochrones(route: gpd.GeoDataFrame | nx.Graph, starting_point: shapely.geometry.Point):
    if type(route) == gpd.GeoDataFrame:
        route = momepy.gdf_to_nx(route, approach='primal', length='length')
    return route

In [20]:
create_isochrones(segment_lines(gdf_lines), get_centroids(gdf_shape).iloc[0])